# Tutorial: Rubin DP0.2 Access

This notebook prepares a real access path from LensForge into Rubin DP0.2 services. It focuses only on discovery: one TAP query that can run from a local machine with a Rubin access token, and one Butler discovery query that is designed for a Rubin notebook environment with the Science Pipelines configured.

## Requirements

- A Rubin access token stored in `ACCESS_TOKEN` or in `LensForge/.secrets/rubin_access_token.txt`
- Network access to `https://data.lsst.cloud/`
- For the Butler section: a Rubin notebook environment or a local environment that already defines `DAF_BUTLER_REPOSITORY_INDEX`

The TAP section is the primary local proof artifact. The Butler section documents the exact query shape LensForge will use when the Rubin environment is available.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd
from lsst.rsp import get_access_token, get_tap_service
from lsst.daf.butler import Butler


In [ ]:
def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists():
            return candidate
    return start

repo_root = find_repo_root(Path.cwd().resolve())
token_file = repo_root / '.secrets' / 'rubin_access_token.txt'
os.environ.setdefault('EXTERNAL_INSTANCE_URL', 'https://data.lsst.cloud/')

if 'ACCESS_TOKEN' not in os.environ and token_file.exists():
    os.environ['ACCESS_TOKEN'] = token_file.read_text().strip()

token = get_access_token()
print({'repo_root': str(repo_root), 'token_file_exists': token_file.exists(), 'has_access_token': bool(token)})
if not token:
    raise RuntimeError(
        'No Rubin access token found. Save one to LensForge/.secrets/rubin_access_token.txt '
        'or export ACCESS_TOKEN before running the live DP0.2 query cells.'
    )


## TAP Discovery From a Local Machine

This is the first live DP0.2 artifact LensForge aims to keep in-repo. It uses the official `lsst-rsp` TAP client and queries `ivoa.ObsCore`, which matches the adapter path described in the proposal.

In [ ]:
tap_service = get_tap_service('tap')
tap_query = '''
SELECT TOP 5
    obs_id,
    obs_collection,
    dataproduct_type,
    calib_level,
    s_ra,
    s_dec
FROM ivoa.ObsCore
'''

tap_results = tap_service.search(tap_query)
tap_table = tap_results.to_table().to_pandas()
tap_table


## Butler Discovery In a Rubin Notebook Environment

The official DP0.2 notebook tutorials use `Butler('dp02', collections='2.2i/runs/DP0.2')`. On a plain laptop this usually still needs Rubin environment metadata such as `DAF_BUTLER_REPOSITORY_INDEX`, so this section is prepared for RSP-style execution rather than claimed as a guaranteed local-laptop path.

In [ ]:
repo_index = os.environ.get('DAF_BUTLER_REPOSITORY_INDEX')
if not repo_index:
    raise RuntimeError(
        'DAF_BUTLER_REPOSITORY_INDEX is not set. Run this cell inside a Rubin notebook environment '
        'or define a valid repository index before attempting Butler discovery.'
    )

butler = Butler('dp02', collections='2.2i/runs/DP0.2')
refs = butler.query_datasets('calexp', data_id={'visit': 192350, 'detector': 175}, limit=5)

butler_table = pd.DataFrame([
    {
        'datasetType': ref.datasetType.name,
        'run': ref.run,
        'dataId': dict(ref.dataId.mapping),
    }
    for ref in refs
])
butler_table


## LensForge Relevance

- The TAP result is the natural future implementation target for the `query` stage.
- The Butler discovery result is the natural future implementation target for the `fetch` stage.
- The downstream `cutout -> preprocess -> package` stages in LensForge remain unchanged once these two upstream adapters are real.